# 05 — Entraînement de production

**Projet** : Support client sur fiches produit orchestré avec LangChain (LCEL) — prompt + retriever + citations

## Objectifs pédagogiques

1. Exécuter le pipeline d'entraînement réel, sans le réécrire dans le notebook.
1. Lire les artefacts produits : passages, métriques, fiche de modèle et configuration résolue.
1. Vérifier la reproductibilité : reconstruire l'index retrouve les mêmes passages.

**Objectifs transverses du dépôt**

- Écrire un générateur de corpus dont la réponse de référence est une phrase du corpus, afin que l'exact match mesure la sélection de la bonne phrase.
- Séparer le vocabulaire des fiches (garantie, délai, montant, panne, compatibilité) pour que la question ait une seule bonne fiche.
- Distinguer une question difficile d'une question ambiguë : la paraphrase s'éloigne du texte, elle ne retire pas l'entité qui identifie la réponse.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.7)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Le pipeline, pas une approximation

Un notebook qui « refait » l'entraînement finit toujours par diverger du code de production. Ici on appelle `TrainPipeline`, exactement comme `python -m src.main mode=train`, en redirigeant ses écritures vers `outputs/notebooks`.

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)

In [ ]:
from src.pipelines.train_pipeline import TrainPipeline

TRAIN_RESULT = TrainPipeline(CONFIG, paths=NB_PATHS).run()
print("statut :", TRAIN_RESULT.status)
for message in TRAIN_RESULT.messages:
    print("-", message)
print()
print("artefacts écrits :")
for artifact in TRAIN_RESULT.artifacts:
    print("  -", Path(artifact).relative_to(PROJECT_ROOT))

In [ ]:
metrics = pd.Series(TRAIN_RESULT.metrics).sort_index()
validation_metrics = metrics[
    metrics.index.str.startswith(
        (
            "val_recall",
            "val_mrr",
            "val_ndcg",
            "val_answer",
            "val_citation",
            "val_abstention",
            "val_latency",
        )
    )
]
print("métriques de validation (la validation surveille, le test jugera) :")
validation_metrics.round(4).to_frame("valeur")

### 1.1 La fiche de modèle

Elle est écrite à côté de l'artefact et dit ce que le fichier sérialisé ne dit pas : quelle configuration, quelles métriques, quelles limites connues.

In [ ]:
from src.models import build_model, load_model
from src.utils.io import read_json

ARTIFACT = NB_PATHS.models_dir / "rag_chain.joblib"
TRAINED = load_model(ARTIFACT, config=CONFIG.model_dump())
# La fiche **archivée** est celle qu'a écrite le pipeline : c'est elle qui porte les métriques du
# run. `TRAINED.model_card()` en reconstruit une vide, exactement comme une instance rechargée ne
# connaît pas l'historique d'entraînement.
card = read_json(NB_PATHS.models_dir / "model_card.json")
for key in (
    "model_name",
    "framework",
    "algorithm",
    "task",
    "n_documents",
    "n_chunks",
    "created_at",
    "artifact",
):
    print(f"  {key:14s}: {card.get(key)}")
print()
print("paramètres effectifs :")
display(pd.Series(card.get("params", {})).to_frame("valeur"))
print(f"métriques archivées : {len(card.get('metrics', {}))}")
print("notes de la fiche :")
for note in card.get("notes", [])[:4]:
    print("  -", note)

In [ ]:
chunks = TRAINED.chunks
display(chunks.head(3)[["chunk_id", "doc_id", "chunk_index", "n_tokens"]])
print(f"{len(chunks)} passages indexés")
print("fichiers de passages :", [path.name for path in NB_PATHS.processed_dir.glob("chunks.*")])
print("taille moyenne d'un passage :", round(float(chunks["n_tokens"].mean()), 1), "tokens")

## 2. Reproductibilité

Reconstruire l'index doit retrouver les mêmes passages et les mêmes scores : c'est la condition pour que deux variantes soient comparables, et pour qu'un incident puisse être rejoué à l'identique.

In [ ]:
rebuilt = build_model(CONFIG.model_dump())
rebuilt.fit(DOCUMENTS)
probe = str(TEST["question"].iloc[0])
first = [passage.chunk_id for passage in TRAINED.retrieve(probe, 5)]
second = [passage.chunk_id for passage in rebuilt.retrieve(probe, 5)]
print("question de sonde :", probe)
print("index entraîné    :", first)
print("index reconstruit :", second)
print("identiques        :", first == second)

In [ ]:
resolved = read_json(NB_PATHS.models_dir / "resolved_config.json")
print("configuration résolue, clés de premier niveau :", sorted(resolved)[:12])
print()
print(
    "Cette copie permet de rejouer un run : l'artefact seul ne dit pas quel override l'a produit."
)

**Ce qu'il faut retenir**

- L'indexation est un `fit` au sens du contrat : elle produit des artefacts, des métriques et une fiche de modèle, comme n'importe quel autre modèle.
- Les passages sont persistés : sans eux, impossible d'expliquer pourquoi une question a échoué, ni de rejouer une évaluation.
- La configuration résolue est archivée avec l'artefact : c'est elle qui rend le résultat reproductible, pas la graine seule.